# Step 18 — HDB Opportunity Analysis

Identify segments for further commercial research using observed resale activity. This is a transparent market-screening framework, not a forecast of returns or a recommendation to buy property.

The matrix compares 2025 volume with 2025 versus 2024 transaction growth. Bubble area represents 2025 transaction value; colour represents median price-per-sqm growth. Historical figures include January 2017–August 2026 and are only used for the inherited sample screen. This is a retrospective 2025 analysis, not a historical backtest: eligibility uses data through August 2026.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from IPython.display import display

PROJECT_ROOT = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data").is_dir() and (path / "notebooks").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the project directory.")

OUTPUT_DIR = PROJECT_ROOT / "outputs"
CHART_DIR = PROJECT_ROOT / "charts"
CHART_DIR.mkdir(parents=True, exist_ok=True)
all_segments = pd.read_csv(OUTPUT_DIR / "hdb_segment_analysis_all.csv")
segments = pd.read_csv(OUTPUT_DIR / "hdb_segment_analysis_filtered.csv")
required = ["transactions_2024", "transactions_2025", "transaction_growth_pct", "price_growth_pct", "transaction_value_2025"]
assert not segments[required].isna().any().any()
assert np.isfinite(segments[required].to_numpy()).all()
assert not segments.duplicated(["town", "flat_type"]).any()
assert segments["transactions_2024"].ge(30).all() and segments["transactions_2025"].ge(30).all()
assert np.allclose(segments["transaction_growth_pct"], (segments["transactions_2025"] / segments["transactions_2024"] - 1) * 100)
print(f"Eligible segments: {len(segments)} of {len(all_segments)}")


## 1. Classification rules

Use the median 2025 transaction count among eligible segments as the size threshold and zero transaction growth as the momentum threshold. Equality at the size boundary counts as large; zero growth counts as stable/declining.

| Category | 2025 volume | Transaction growth | Research use |
|---|---|---|---|
| Priority | At or above median | Above zero | Investigate scale plus expanding activity |
| Emerging | Below median | Above zero | Monitor smaller expanding segments |
| Established | At or above median | Zero or below | Research large markets despite weaker activity |
| Lower Priority | Below median | Zero or below | Lower priority within this volume-growth framework |

Price growth is supplementary context and does not determine the category. Negative price growth is flagged for review. No weighted opportunity score is used: within categories, sort by 2025 transaction volume, then transaction growth. Threshold choices are judgement calls, so a sensitivity check follows.


In [2]:
SIZE_THRESHOLD = float(segments["transactions_2025"].median())
large = segments["transactions_2025"].ge(SIZE_THRESHOLD)
growing = segments["transaction_growth_pct"].gt(0)
segments["opportunity_category"] = np.select(
    [large & growing, ~large & growing, large & ~growing],
    ["Priority", "Emerging", "Established"], default="Lower Priority"
)
segments["price_momentum"] = np.where(segments["price_growth_pct"].gt(0), "Positive", "Flat or negative")
segments["composition_review"] = segments["segment"].isin([
    "CENTRAL AREA — 4 ROOM", "CLEMENTI — 4 ROOM", "CLEMENTI — 5 ROOM"
])
segments["transaction_value_2025_million"] = segments["transaction_value_2025"] / 1_000_000
print(f"Large-segment threshold: {SIZE_THRESHOLD:,.0f} transactions in 2025")
market_2024 = all_segments["transactions_2024"].sum()
market_2025 = all_segments["transactions_2025"].sum()
market_growth = (market_2025 / market_2024 - 1) * 100
print(f"Whole-dataset 2025 activity: {market_2025:,.0f} transactions; growth: {market_growth:.2f}%")
print(f"Eligible segments account for {segments['transactions_2025'].sum() / market_2025:.1%} of 2025 transactions.")
order = ["Priority", "Emerging", "Established", "Lower Priority"]
summary = segments.groupby("opportunity_category").agg(
    segment_count=("segment", "size"),
    transactions_2024=("transactions_2024", "sum"),
    transactions_2025=("transactions_2025", "sum"),
    transaction_value_2025=("transaction_value_2025", "sum"),
    median_segment_price_growth_pct=("price_growth_pct", "median")
).reindex(order)
summary["aggregate_transaction_growth_pct"] = (summary["transactions_2025"] / summary["transactions_2024"] - 1) * 100
summary["share_of_eligible_2025_transactions_pct"] = summary["transactions_2025"] / segments["transactions_2025"].sum() * 100
display(summary.round(2))


Large-segment threshold: 209 transactions in 2025
Whole-dataset 2025 activity: 25,085 transactions; growth: -9.87%
Eligible segments account for 95.6% of 2025 transactions.


,segment_count,transactions_2024,transactions_2025,transaction_value_2025,median_segment_price_growth_pct,aggregate_transaction_growth_pct,share_of_eligible_2025_transactions_pct
opportunity_category,,,,,,,
Priority,9,3658.0,4036.0,2.736780e+09,8.63,10.33,16.82
Emerging,11,1083.0,1236.0,1.018845e+09,6.07,14.13,5.15
Established,35,17765.0,15226.0,9.411029e+09,5.75,-14.29,63.47
Lower Priority,30,4244.0,3493.0,2.573836e+09,5.93,-17.70,14.56


## 2. Opportunity matrix

Dashed lines show the classification boundaries. All eligible segments are plotted; selected large and high-growth segments are labelled to keep the chart readable.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 9))
values = segments["transaction_value_2025"]
area_scale = 1800 / values.max()
colour_limit = max(abs(segments["price_growth_pct"]).max(), 1)
points = ax.scatter(
    segments["transactions_2025"], segments["transaction_growth_pct"],
    s=values * area_scale, c=segments["price_growth_pct"],
    cmap="coolwarm", norm=TwoSlopeNorm(vmin=-colour_limit, vcenter=0, vmax=colour_limit),
    alpha=0.78, edgecolors="#334155", linewidths=0.5
)
ax.axvline(SIZE_THRESHOLD, color="#475569", linestyle="--", linewidth=1)
ax.axhline(0, color="#475569", linestyle="--", linewidth=1)
ax.set_xlim(0, segments["transactions_2025"].max() * 1.2)
ax.set_ylim(-45, 72)
for label, xy in [("Emerging", (0.02, .97)), ("Priority", (.7, .97)), ("Lower Priority", (.02, .03)), ("Established", (.7, .03))]:
    ax.text(*xy, label, transform=ax.transAxes, fontsize=12, fontweight="bold", color="#334155", va="top" if xy[1]>.5 else "bottom")
selected = pd.concat([
    segments.loc[segments["opportunity_category"].eq("Priority")].nlargest(3, "transactions_2025"),
    segments.nlargest(2, "transactions_2025"),
    segments.nlargest(1, "transaction_growth_pct"),
    segments.loc[segments["segment"].eq("TOA PAYOH — 4 ROOM")]
]).drop_duplicates("segment")
for j, (_, row) in enumerate(selected.iterrows()):
    ax.annotate(row["segment"].title(), (row["transactions_2025"], row["transaction_growth_pct"]),
                xytext=(8, 12 if j % 2 == 0 else -18), textcoords="offset points", fontsize=8,
                arrowprops=dict(arrowstyle="-", color="#64748b", lw=.6))
fig.colorbar(points, ax=ax, label="Median price-per-sqm growth, 2025 vs 2024 (%)", shrink=.8)
for million in [100, 300, 600]:
    ax.scatter([], [], s=million * 1_000_000 * area_scale, c="#cbd5e1", edgecolors="#334155", label=f"S${million}m")
ax.legend(title="2025 transaction value", loc="upper right", bbox_to_anchor=(1, .88), framealpha=.95, labelspacing=2, borderpad=1.4)
ax.set_title("HDB Segment Opportunity Matrix — 2025", fontsize=17, pad=15)
ax.set_xlabel(f"2025 transactions (size threshold: {SIZE_THRESHOLD:,.0f})")
ax.set_ylabel("Transaction growth, 2025 vs 2024 (%)")
ax.grid(alpha=.15)
fig.tight_layout()
fig.savefig(CHART_DIR / "hdb_opportunity_matrix.png", dpi=180, bbox_inches="tight")
plt.show()


## 3. Research shortlist by category

Tables show the ten largest segments in each category, ordered by 2025 volume. A composition flag refers to the three segments investigated in Step 17; an unflagged segment is not proof of unchanged composition.

In [4]:
columns = ["segment", "transactions_2024", "transactions_2025", "transaction_growth_pct", "price_growth_pct", "transaction_value_2025_million", "composition_review"]
ranked = segments.sort_values(["transactions_2025", "transaction_growth_pct"], ascending=False)
for category in order:
    print(category)
    display(ranked.loc[ranked["opportunity_category"].eq(category), columns].head(10).round(2))


Priority
Emerging
Established
Lower Priority


,segment,transactions_2024,transactions_2025,transaction_growth_pct,price_growth_pct,transaction_value_2025_million,composition_review
71,TAMPINES — 4 ROOM,867.0,892.0,2.88,6.86,611.28,False
58,SEMBAWANG — 4 ROOM,485.0,545.0,12.37,8.70,342.29,False
72,TAMPINES — 5 ROOM,506.0,509.0,0.59,11.74,419.69,False
74,TOA PAYOH — 3 ROOM,366.0,424.0,15.85,4.98,203.98,False
75,TOA PAYOH — 4 ROOM,299.0,423.0,41.47,12.90,386.18,False
17,BUKIT PANJANG — 4 ROOM,370.0,383.0,3.51,8.63,225.26,False
32,HOUGANG — 3 ROOM,308.0,326.0,5.84,6.58,149.41,False
59,SEMBAWANG — 5 ROOM,256.0,325.0,26.95,8.18,225.89,False
27,CLEMENTI — 4 ROOM,201.0,209.0,3.98,30.94,172.80,True


,segment,transactions_2024,transactions_2025,transaction_growth_pct,price_growth_pct,transaction_value_2025_million,composition_review
48,PASIR RIS — 5 ROOM,195.0,204.0,4.62,4.56,150.71,False
76,TOA PAYOH — 5 ROOM,108.0,172.0,59.26,0.52,183.38,False
73,TAMPINES — EXECUTIVE,150.0,154.0,2.67,5.33,150.67,False
45,KALLANG/WHAMPOA — 5 ROOM,111.0,117.0,5.41,5.85,117.64,False
57,SEMBAWANG — 3 ROOM,111.0,113.0,1.80,7.22,59.56,False
61,SENGKANG — 2 ROOM,65.0,90.0,38.46,6.07,34.36,False
46,MARINE PARADE — 3 ROOM,78.0,86.0,10.26,6.22,41.77,False
28,CLEMENTI — 5 ROOM,66.0,84.0,27.27,24.37,87.34,True
60,SEMBAWANG — EXECUTIVE,71.0,77.0,8.45,8.14,57.61,False
31,GEYLANG — 5 ROOM,61.0,71.0,16.39,-0.27,62.44,False


,segment,transactions_2024,transactions_2025,transaction_growth_pct,price_growth_pct,transaction_value_2025_million,composition_review
63,SENGKANG — 4 ROOM,1040.0,940.0,-9.62,5.57,619.35,False
78,WOODLANDS — 4 ROOM,882.0,836.0,-5.22,4.02,472.16,False
82,YISHUN — 4 ROOM,894.0,825.0,-7.72,4.68,467.23,False
52,PUNGGOL — 4 ROOM,1005.0,775.0,-22.89,5.93,530.61,False
64,SENGKANG — 5 ROOM,717.0,642.0,-10.46,6.47,468.96,False
40,JURONG WEST — 4 ROOM,764.0,609.0,-20.29,6.99,338.36,False
33,HOUGANG — 4 ROOM,671.0,597.0,-11.03,7.75,376.18,False
10,BUKIT BATOK — 4 ROOM,810.0,548.0,-32.35,3.22,339.73,False
0,ANG MO KIO — 3 ROOM,571.0,535.0,-6.30,6.33,243.38,False
79,WOODLANDS — 5 ROOM,626.0,515.0,-17.73,4.57,346.43,False


,segment,transactions_2024,transactions_2025,transaction_growth_pct,price_growth_pct,transaction_value_2025_million,composition_review
51,PUNGGOL — 3 ROOM,259.0,203.0,-21.62,7.00,110.46,False
77,WOODLANDS — 3 ROOM,271.0,201.0,-25.83,5.88,88.20,False
67,SERANGOON — 4 ROOM,197.0,186.0,-5.58,5.70,126.80,False
36,JURONG EAST — 3 ROOM,230.0,182.0,-20.87,5.77,76.19,False
7,BISHAN — 4 ROOM,187.0,178.0,-4.81,4.24,139.78,False
5,BEDOK — 5 ROOM,252.0,178.0,-29.37,2.17,139.00,False
15,BUKIT MERAH — 5 ROOM,183.0,167.0,-8.74,10.92,178.39,False
80,WOODLANDS — EXECUTIVE,214.0,155.0,-27.57,5.89,142.71,False
62,SENGKANG — 3 ROOM,171.0,153.0,-10.53,9.16,82.88,False
37,JURONG EAST — 4 ROOM,171.0,144.0,-15.79,5.98,80.72,False


## 4. Sensitivity to the size threshold

Keep growth above zero and compare the 40th, 50th, and 60th percentiles for size. Segments that remain Priority at all three thresholds form a more stable scale-and-growth shortlist.

In [5]:
sensitivity = []
priority_sets = []
for quantile in [.4, .5, .6]:
    cutoff = segments["transactions_2025"].quantile(quantile)
    is_priority = segments["transactions_2025"].ge(cutoff) & segments["transaction_growth_pct"].gt(0)
    priority_sets.append(set(segments.loc[is_priority, "segment"]))
    sensitivity.append({"size_percentile": quantile * 100, "transaction_cutoff": cutoff, "priority_segments": int(is_priority.sum())})
sensitivity = pd.DataFrame(sensitivity)
display(sensitivity)
stable_priority = set.intersection(*priority_sets)
segments["priority_across_size_thresholds"] = segments["segment"].isin(stable_priority)
display(segments.loc[segments["priority_across_size_thresholds"], columns].sort_values("transactions_2025", ascending=False).round(2))


,size_percentile,transaction_cutoff,priority_segments
0,40.0,175.6,10
1,50.0,209.0,9
2,60.0,273.0,8


,segment,transactions_2024,transactions_2025,transaction_growth_pct,price_growth_pct,transaction_value_2025_million,composition_review
71,TAMPINES — 4 ROOM,867.0,892.0,2.88,6.86,611.28,False
58,SEMBAWANG — 4 ROOM,485.0,545.0,12.37,8.70,342.29,False
72,TAMPINES — 5 ROOM,506.0,509.0,0.59,11.74,419.69,False
74,TOA PAYOH — 3 ROOM,366.0,424.0,15.85,4.98,203.98,False
75,TOA PAYOH — 4 ROOM,299.0,423.0,41.47,12.90,386.18,False
17,BUKIT PANJANG — 4 ROOM,370.0,383.0,3.51,8.63,225.26,False
32,HOUGANG — 3 ROOM,308.0,326.0,5.84,6.58,149.41,False
59,SEMBAWANG — 5 ROOM,256.0,325.0,26.95,8.18,225.89,False


## 5. Export complete results

All 131 segments are retained in the full output; ineligible rows are explicitly marked and receive no opportunity category.

In [ ]:
classification = segments[["town", "flat_type", "opportunity_category", "price_momentum", "composition_review", "priority_across_size_thresholds"]]
full_output = all_segments.merge(classification, on=["town", "flat_type"], how="left", validate="one_to_one")
full_output["classification_status"] = np.where(full_output["opportunity_category"].notna(), "Eligible", "Excluded by sample screen")
assert len(full_output) == len(all_segments)
assert summary["segment_count"].sum() == len(segments)
segments.to_csv(OUTPUT_DIR / "hdb_opportunity_segments.csv", index=False)
full_output.to_csv(OUTPUT_DIR / "hdb_opportunity_all_segments.csv", index=False)
summary.to_csv(OUTPUT_DIR / "hdb_opportunity_category_summary.csv")
sensitivity.to_csv(OUTPUT_DIR / "hdb_opportunity_sensitivity.csv", index=False)
print("Saved opportunity tables to", OUTPUT_DIR, "and chart to", CHART_DIR)


## 6. Interpretation and limitations

Priority denotes scale and positive observed activity within this framework. Large declining markets can still matter commercially, and price growth can reflect a different mix of flats sold. The 2024–2025 comparison is a single-year change, not a forecast. Transaction value is property sales value, not agency revenue or profit. Further research should check property composition, multi-year trends, and the specific business objective before making decisions.


## 7. Verified findings

### Method

A volume-growth matrix classified 85 eligible Town × Flat Type segments using 2025 transaction counts and 2025 versus 2024 transaction growth. Large segments had at least 209 transactions, the eligible-segment median. Positive growth meant strictly above zero. Bubble area represented 2025 transaction value and colour represented median price-per-sqm growth.

Eligibility required at least 500 historical transactions and 30 transactions in each comparison year. Historical eligibility uses January 2017–August 2026, making this a retrospective analysis rather than an as-of-2025 backtest. The eligible segments account for 95.6% of 2025 transactions in the dataset.

### Classification results

| Category | Segments | Interpretation |
|---|---:|---|
| Priority | 9 | Larger segments with growing transaction activity |
| Emerging | 11 | Smaller segments with growing transaction activity |
| Established | 35 | Larger segments with flat or declining transaction activity |
| Lower Priority | 30 | Smaller segments with flat or declining transaction activity |

The nine Priority segments recorded 4,036 transactions and approximately S$2.737 billion in transaction value in 2025. Their aggregate volume grew 10.33%, while total dataset volume declined 9.87%.

### Priority research candidates

- Tampines × 4 Room had the largest 2025 volume within Priority: 892 transactions, +2.88% transaction growth and +6.86% median price-per-sqm growth.
- Sembawang × 4 Room recorded 545 transactions, +12.37% transaction growth and +8.70% price-per-sqm growth.
- Toa Payoh × 4 Room recorded 423 transactions, +41.47% transaction growth and +12.90% price-per-sqm growth.
- Sembawang × 5 Room recorded 325 transactions, +26.95% transaction growth and +8.18% price-per-sqm growth.
- Tampines × 5 Room qualifies on size and positive growth, but its +0.59% activity growth should be described as approximately stable rather than strong momentum.

### Robustness and interpretation

Changing the size cutoff to the 40th, 50th and 60th percentiles produced 10, 9 and 8 Priority segments. Eight remained Priority throughout: Tampines × 4 Room, Sembawang × 4 Room, Tampines × 5 Room, Toa Payoh × 3 Room, Toa Payoh × 4 Room, Bukit Panjang × 4 Room, Hougang × 3 Room and Sembawang × 5 Room.

Clementi × 4 Room sits exactly at the baseline 209-transaction size boundary and leaves Priority under the stricter cutoff. Its large median price increase coincides with a shift toward longer remaining leases in Step 17's composition check.

There are 20 activity-growing segments across Priority and Emerging, compared with 19 segments showing both positive activity and positive price growth. The difference is Geylang × 5 Room: transaction growth was +16.39% while median price-per-sqm growth was -0.27%.

These categories prioritize commercial research; they do not forecast investment returns, agency revenue or profitability. Median price changes can reflect the mix of properties sold. Large Established segments still account for 63.47% of eligible 2025 transactions and should not be ignored simply because activity declined.
